In [ ]:
import pandas as pd
fn = '../data/raw/jstor_metadata_2025-11-28.jsonl.gz'

In [ ]:
import sys
sys.path.append('..')
from osp import *

In [12]:
dfm = get_corpus_metadata().query('discipline == "Philosophy"')
dfm

,uuid,title,author,year,journal,volume,issue,url,publisher,discipline,decade,period,century,halfcentury,century_discipline,halfcentury_discipline,period_discipline,decade_discipline,century_journal,journal_orig
id,,,,,,,,,,,,,,,,,,,,
phil/10.2307/40231690,f6eecd30-8c4a-3c3d-a9da-4e777d091b2c,"""Aristotlés"" Horror Vacui",John Thorp,1990,Canadian Journal of Philosophy,20,2,jstor.org/stable/10.2307/40231690,Cambridge University Press,Philosophy,1990,1975-2000,C20,lC20,C20 Philosophy,lC20 Philosophy,1975-2000 Philosophy,1990 Philosophy,C20 Canadian Journal of Philosophy,Canadian Journal of Philosophy
phil/10.2307/40230399,aaff574b-da8a-389b-b82c-8bcfbf981a8e,"""None in Particular""",John Woods,1973,Canadian Journal of Philosophy,2,3,jstor.org/stable/10.2307/40230399,Cambridge University Press,Philosophy,1970,1950-1975,C20,lC20,C20 Philosophy,lC20 Philosophy,1950-1975 Philosophy,1970 Philosophy,C20 Canadian Journal of Philosophy,Canadian Journal of Philosophy
phil/10.2307/40231533,9a99d15f-bbe2-349e-b5af-cf96fd35c6eb,"""Tractatus"" 2.022-2.023",Raymond D. Bradley,1987,Canadian Journal of Philosophy,17,2,jstor.org/stable/10.2307/40231533,Cambridge University Press,Philosophy,1980,1975-2000,C20,lC20,C20 Philosophy,lC20 Philosophy,1975-2000 Philosophy,1980 Philosophy,C20 Canadian Journal of Philosophy,Canadian Journal of Philosophy
phil/10.2307/40230622,9af0da08-8130-3178-86cb-5df65d6fb0cd,"""Tractatus"" 5.54-5.5422",Eric B. Dayton,1976,Canadian Journal of Philosophy,6,2,jstor.org/stable/10.2307/40230622,Cambridge University Press,Philosophy,1970,1975-2000,C20,lC20,C20 Philosophy,lC20 Philosophy,1975-2000 Philosophy,1970 Philosophy,C20 Canadian Journal of Philosophy,Canadian Journal of Philosophy
phil/10.2307/40231225,041ecb2f-f8ab-398e-a1df-8813eaffa841,"'Can,' Compatibilism, and Possible Worlds",Michael J. Zimmerman,1981,Canadian Journal of Philosophy,11,4,jstor.org/stable/10.2307/40231225,Cambridge University Press,Philosophy,1980,1975-2000,C20,lC20,C20 Philosophy,lC20 Philosophy,1975-2000 Philosophy,1980 Philosophy,C20 Canadian Journal of Philosophy,Canadian Journal of Philosophy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
phil/10.2307/20130578,af8cacaf-c711-3d73-a7e0-7bef979addb9,Wrongs and Faults,John Gardner,2005,The Review of Metaphysics,59,1,jstor.org/stable/10.2307/20130578,Philosophy Education Society Inc.,Philosophy,2000,2000-2025,C21,eC21,C21 Philosophy,eC21 Philosophy,2000-2025 Philosophy,2000 Philosophy,C21 The Review of Metaphysics,The Review of Metaphysics
phil/10.2307/20123866,b338bd25-39da-348f-87d4-b5d5adc6a566,Yoga: The Path to Freedom from Suffering,K. Satchidananda Murty,1961,The Review of Metaphysics,15,1,jstor.org/stable/10.2307/20123866,Philosophy Education Society Inc.,Philosophy,1960,1950-1975,C20,lC20,C20 Philosophy,lC20 Philosophy,1950-1975 Philosophy,1960 Philosophy,C20 The Review of Metaphysics,The Review of Metaphysics
phil/10.2307/20123953,ada1c27d-6cc4-3bfc-ac1a-e7f157b78170,Zeno's Paradoxes on Motion,John O. Nelson,1963,The Review of Metaphysics,16,3,jstor.org/stable/10.2307/20123953,Philosophy Education Society Inc.,Philosophy,1960,1950-1975,C20,lC20,C20 Philosophy,lC20 Philosophy,1950-1975 Philosophy,1960 Philosophy,C20 The Review of Metaphysics,The Review of Metaphysics


In [ ]:
# dfm.year.value_counts()

In [ ]:
# dfall = pd.read_json('../data/raw/jstor_metadata_2025-11-28.jsonl.gz',lines=True)
# dfall

In [21]:
from tqdm import tqdm
import json
import orjsonl

def iter_jsonl(fn):
    yield from orjsonl.stream(fn)

def search_jsonl(fn, total=12412004, philosophy=True):
    for d in tqdm(iter_jsonl(fn), total=total):
        if d.get("c5_data_type") == "Journal" and d.get('content_subtype') == "research-article":
            disciplines = d.get('discipline_names',[])
            if not disciplines:
                continue
            is_phil = 'Philosophy' in disciplines
            odx = {'url': d.get('url'), 'published_date': d.get('published_date'), 'item_id': d.get('item_id'), 'discipline_names': disciplines}
            if is_phil and philosophy:
                yield odx
            elif not is_phil and not philosophy:
                yield odx

In [22]:
import pandas as pd
df_nonphil = pd.DataFrame(search_jsonl(fn, philosophy=False))

100%|██████████| 12412004/12412004 [00:45<00:00, 275418.64it/s]


In [23]:
df_nonphil['year'] = df_nonphil['published_date'].str[:4]

In [24]:
year_counts_dfm = dfm.year.value_counts().sort_index()
year_counts_dfm

year
1900      60
1901      58
1902      57
1903      56
1904      67
        ... 
2017     559
2018     559
2019     564
2020     374
2021    1644
Name: count, Length: 122, dtype: int64

In [25]:
def sample_nonphil(df_nonphil, year_counts_dfm):
    o=[]
    for y,c in tqdm(list(year_counts_dfm.items())):
        dfy = df_nonphil[df_nonphil.year == str(y)]
        o.append(dfy.sample(c if c < len(dfy) else len(dfy)))
    return pd.concat(o)

df_nonphil_sampled = sample_nonphil(df_nonphil, year_counts_dfm)
df_nonphil_sampled

100%|██████████| 122/122 [00:33<00:00,  3.60it/s]


,url,published_date,item_id,discipline_names,year
97260,www.jstor.org/stable/10.2307/20544715,1900-05-1,cef2b74a-5f81-38e8-a1a0-ee870316ca43,[Irish Studies],1900
5741745,www.jstor.org/stable/10.2307/783362,1900-12-0,1424ff7f-2a0c-388f-81af-3a8c832ef577,[Law],1900
3417546,www.jstor.org/stable/10.2307/4062670,1900-01-0,77f5d0fa-6343-3bd4-a43e-aafc2b41524b,[Biological Sciences],1900
5700387,www.jstor.org/stable/10.2307/1098424,1900-10-0,e5c5e390-b66c-3c0d-9304-44f0d4e028dc,[Law],1900
5247681,www.jstor.org/stable/10.2307/20499607,1900-06-0,41c13c48-87a4-384c-a133-7400f8a3ee37,"[Religion, Language & Literature, Irish Studies, History]",1900
...,...,...,...,...,...
2341847,www.jstor.org/stable/10.2307/27351376,2021-12-0,385d2590-c3cc-3d64-9ccf-aed313244deb,[Health Sciences],2021
3517159,www.jstor.org/stable/10.2307/27075864,2021-09-2,53663faa-b550-301c-ac83-e0f7cc52120c,"[General Science, Biological Sciences]",2021
3274604,www.jstor.org/stable/10.2307/27296356,2021-01-0,b5eb8f1b-4db7-3d48-a346-76d45b7d42a4,"[Biological Sciences, Ecology & Evolutionary Biology, Botany & Plant Sciences]",2021
790157,www.jstor.org/stable/10.2307/27168500,2021-01-0,2d4f27c1-cfe5-34a6-ae08-22de7db253e6,[Law],2021


In [26]:
dfm.year.value_counts().sort_index()

year
1900      60
1901      58
1902      57
1903      56
1904      67
        ... 
2017     559
2018     559
2019     564
2020     374
2021    1644
Name: count, Length: 122, dtype: int64

In [28]:
df_nonphil_sampled.year.value_counts().sort_index()

year
1900      60
1901      58
1902      57
1903      56
1904      67
        ... 
2017     559
2018     559
2019     564
2020     374
2021    1644
Name: count, Length: 122, dtype: int64

In [31]:
df_nonphil_sampled.discipline_names.value_counts().head(25)

discipline_names
[Language & Literature]                                                              2656
[Health Sciences]                                                                    2044
[Law]                                                                                2024
[General Science, Biological Sciences]                                               1717
[Mathematics]                                                                         863
[Education]                                                                           712
[Military Studies, Peace & Conflict Studies, Political Science, Security Studies]     606
[Art & Art History]                                                                   600
[Botany & Plant Sciences]                                                             514
[Religion]                                                                            460
[Statistics]                                                                       

discipline_names
[Health Sciences]                                                                                    727
[Law]                                                                                                697
[Language & Literature]                                                                              417
[General Science, Biological Sciences]                                                               263
[Art & Art History]                                                                                  241
[History, American Studies]                                                                          141
[Religion]                                                                                           138
[Classical Studies]                                                                                  133
[Mathematics]                                                                                        126
[Education]                           

In [40]:
s1=df_nonphil_sampled.query('year < "1950"').discipline_names.value_counts()
s2=df_nonphil_sampled.query('year > "1950"').discipline_names.value_counts()

(s1/s1.sum()).head(10) * 100

discipline_names
[Health Sciences]                         12.020503
[Law]                                     11.524471
[Language & Literature]                    6.894841
[General Science, Biological Sciences]     4.348545
[Art & Art History]                        3.984788
[History, American Studies]                2.331349
[Religion]                                 2.281746
[Classical Studies]                        2.199074
[Mathematics]                              2.083333
[Education]                                1.868386
Name: count, dtype: float64

In [41]:
(s2/s2.sum()).head(10) * 100

discipline_names
[Language & Literature]                                                              8.535552
[General Science, Biological Sciences]                                               5.569679
[Law]                                                                                4.995763
[Health Sciences]                                                                    4.991911
[Mathematics]                                                                        2.788691
[Education]                                                                          2.284108
[Military Studies, Peace & Conflict Studies, Political Science, Security Studies]    2.260997
[Botany & Plant Sciences]                                                            1.656267
[Statistics]                                                                         1.610045
[Art & Art History]                                                                  1.367383
Name: count, dtype: float64

In [45]:
ids2 = df_nonphil_sampled.item_id.sample(frac=1).tolist()

In [46]:
with open('../data/jstor_ids_nonphil.txt','w') as of:
    of.write('\n'.join(ids2))

In [47]:
!wc -l ../data/jstor_ids*

   32782 ../data/jstor_ids.txt
   32276 ../data/jstor_ids_nonphil.txt
   65058 total


In [ ]:
discs = Counter()
for d in tqdm(iter_jsonl(fn)):
    if d.get("c5_data_type") == "Journal" and d.get('content_subtype') == "research-article": 
        disc = d.get('discipline_names')
        if disc:
            discs[' & '.join(sorted(disc))] += 1
discs

In [ ]:
next(iter_jsonl(fn))

In [ ]:
ld = list(search_jsonl(fn))
df = pd.DataFrame(ld)
len(df)

In [ ]:
df.content_subtype.value_counts()

In [ ]:
dfx = df[df.content_subtype=="research-article"]

In [ ]:
dict(dfx.iloc[0])

In [ ]:
all_collections = Counter([c for cs in dfx.discipline_names.dropna() for c in cs])

In [ ]:
all_collections2 = Counter([tuple(cs) for cs in dfx.discipline_names.dropna()])
all_collections2

In [ ]:
[x for x in jcounts.index if 'Speculative' in x]

In [ ]:
# df['date'] = pd.to_datetime(df['published_date'], errors='coerce')
df['year'] = df['published_date'].fillna('').apply(lambda x: x.split('-')[0] if isinstance(x, str) else '0').apply(int)
df['decade'] = df['year'] // 10 * 10

In [ ]:
cdf=df.groupby('is_part_of').decade.value_counts()
cdf.columns=['count']
cdf = cdf.reset_index().pivot(columns='decade',index='is_part_of',values='count').fillna(0).applymap(int)
cdf['total'] = cdf.sum(axis=1)
cdf = cdf.sort_values('total',ascending=False)
cdf

In [ ]:
# !pip install openpyxl
# cdf.to_excel('../data/philosophy_journals_article_counts.xlsx')

In [ ]:
import pandas as pd
cdf2 = pd.read_excel('../data/philosophy_journals_article_counts-selected.xlsx')

In [ ]:
jrnls = cdf2[cdf2.selected=="y"].journal.tolist()
jrnls

In [ ]:
# jrnl = ['The Philosophical Review', 'The Journal of Philosophy']
# jrnl = ['Research in Phenomenology','Philosophy and Phenomenological Research']
# jrnl = ['The Journal of Speculative Philosophy']
jdf = df[df.is_part_of.isin(jrnls)]
jdf.decade.value_counts().sort_index()

In [ ]:
jdf.content_subtype.value_counts()

In [ ]:
finaldf = jdf[jdf.content_subtype=="research-article"]

In [ ]:
item_ids = list(finaldf.item_id)
with open('../data/jstor_ids.txt','w') as of:
    of.write('\n'.join(item_ids)) 